In [1]:
#Bu notebookda 3. eğitilen yolov8m modelinin takip testi videosu olacaktır.
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.pt') or f.endswith('.mp4') or f.endswith('.avi'):
            print(os.path.join(root, f)) 

/kaggle/input/datasets/zlemdemir/bantl-video/video_for_model_3.mp4
/kaggle/input/datasets/zlemdemir/3-model/3_model.pt


In [2]:
!pip install ultralytics opencv-python numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.3 MB/s eta 0:00:00a 0:00:01


In [4]:
import cv2
import numpy as np
import json
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
import os

#
input_video = "/kaggle/input/datasets/zlemdemir/bantl-video/video_for_model_3.mp4"
output_video = "/kaggle/working/kesilmis_video.mp4"

# -ss 00:12:00 -> 12. dakikadan başla
# -c copy -> kaliteyi bozmadan çok hızlı kes
# -y -> eski trimmed_video.mp4 varsa sormadan üzerine yaz
os.system(f"ffmpeg -i {input_video} -ss 00:12:00 -c copy {output_video} -y")

print("Videonun son 2 dakikası alındı!")

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

Videonun son 2 dakikası alındı!


frame= 2850 fps=1571 q=-1.0 Lsize=   14375kB time=00:01:59.95 bitrate= 981.7kbits/s speed=66.1x    
video:12417kB audio:1875kB subtitle:0kB other streams:0kB global headers:0kB muxing overhead: 0.579212%


In [5]:

video_path = '/kaggle/working/kesilmis_video.mp4' 
model_path = '/kaggle/input/datasets/zlemdemir/3-model/3_model.pt'
output_video_path = '/kaggle/working/zone_tracking_output.mp4'
json_output_path = '/kaggle/working/zone_analysis.json'

# Model 
model = YOLO(model_path)

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))


fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

.
zone_polygon = np.array([
    [470, 430],  
    [560, 720],  
    [180, 720], 
    [280, 580]   
], np.int32)
zone_polygon = zone_polygon.reshape((-1, 1, 2))

person_stats = {}

print("Video işleniyor, lütfen bekleyin...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
        
    results = model.track(frame, tracker='botsort.yaml', persist=True, conf=0.5, verbose=False)
    
    # Yasaklı bölgeyi ekrana çiziyoruz
    cv2.polylines(frame, [zone_polygon], isClosed=True, color=(255, 0, 0), thickness=3)
    
    overlay = frame.copy()
    cv2.fillPoly(overlay, [zone_polygon], (255, 0, 0))
    cv2.addWeighted(overlay, 0.2, frame, 0.8, 0, frame)

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy().astype(int)
        
        for box, track_id in zip(boxes, ids):
            x1, y1, x2, y2 = map(int, box)
            
            cx = int((x1 + x2) / 2)
            cy = int(y2)
            
            
            if track_id not in person_stats:
                person_stats[track_id] = {"frames_in": 0, "frames_out": 0}

            is_inside = cv2.pointPolygonTest(zone_polygon, (cx, cy), measureDist=False) >= 0
            
            if is_inside:
                person_stats[track_id]["frames_in"] += 1
                color = (0, 0, 255)  
                label = f"ID: {track_id} (IN)"
            else:
                person_stats[track_id]["frames_out"] += 1
                color = (0, 255, 0)  
                label = f"ID: {track_id} (OUT)"
                
            # Bounding box ve referans noktasını çiz
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.circle(frame, (cx, cy), 5, color, -1) # Ayak referans noktası
            
            # ID ve Durum etiketini yaz
            cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    
    out.write(frame)

cap.release()
out.release()
print(f"Video işleme tamamlandı. Çıktı kaydedildi: {output_video_path}")

Video işleniyor, lütfen bekleyin...
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 425ms
Prepared 1 package in 112ms
Installed 1 package in 9ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 1.3s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Video işleme tamamlandı. Çıktı kaydedildi: /kaggle/working/zone_tracking_output.mp4


In [ ]:
import json

id_mapping = {
    1: [1, 9, 22],
    2: [12, 21, 23],
    3: [19]
}

final_analysis = {}

for new_id, source_ids in id_mapping.items():
    frames_in = 0
    frames_out = 0
    
   
    for s_id in source_ids:
        if s_id in person_stats:
            frames_in += person_stats[s_id]["frames_in"]
            frames_out += person_stats[s_id]["frames_out"]
            
    total_frames = frames_in + frames_out
    
    
    if total_frames == 0:
        continue
        
    
    time_in_sec = frames_in / fps
    time_out_sec = frames_out / fps
    total_time_sec = total_frames / fps
    
    
    ratio_in = frames_in / total_frames
    ratio_out = frames_out / total_frames
    
    final_analysis[f"ID_{new_id}"] = {
        "toplam_sure_saniye": round(total_time_sec, 2),
        "bolge_ici_sure_saniye": round(time_in_sec, 2),
        "bolge_disi_sure_saniye": round(time_out_sec, 2),
        "bolge_ici_oran": round(ratio_in, 3),
        "bolge_disi_oran": round(ratio_out, 3),
        "birlestirilen_ham_IDler": source_ids  # Kontrol amaçlı hangi ID'lerin birleştiğini rapora ekliyoruz
    }

# JSON dosyasına kaydetme
with open(json_output_path, 'w', encoding='utf-8') as f:
    json.dump(final_analysis, f, ensure_ascii=False, indent=4)

print(f"Analiz sonuçları birleştirilerek JSON olarak kaydedildi: {json_output_path}")

# Ekrana bir önizleme yazdır
print("\n---ANALİZ SONUÇLARI ÖZETİ---")
print(json.dumps(final_analysis, ensure_ascii=False, indent=4))